In [315]:
import pandas as pd
import yfinance as yf


In [321]:
CBA_Stock = pd.read_csv("./data/CBA_target_ready_2015_present.csv", parse_dates=["Date"])

print(CBA_Stock.head())
print(CBA_Stock.tail())
print(CBA_Stock.shape)
print(CBA_Stock.columns)


        Date       Open       High        Low      Close  Adj Close   Volume  \
0 2015-01-02  84.959686  85.277962  84.661308  85.277962  51.295269   949439   
1 2015-01-05  85.238182  85.775269  85.049202  85.486832  51.420902  1351651   
2 2015-01-06  84.641411  85.337639  84.412651  84.840332  51.032032  2477275   
3 2015-01-07  84.850281  85.029312  84.094376  84.651360  50.918358  2127190   
4 2015-01-08  85.079041  85.188446  84.671249  84.929848  51.085873  1997761   

   Target_t1  Target_t7  
0          1          0  
1          0          0  
2          0          0  
3          1          0  
4          1          0  
           Date        Open        High         Low       Close   Adj Close  \
2851 2026-04-09  180.570007  182.529999  180.470001  182.529999  182.529999   
2852 2026-04-10  181.000000  183.539993  181.000000  183.380005  183.380005   
2853 2026-04-13  183.050003  184.940002  182.619995  183.199997  183.199997   
2854 2026-04-14  185.000000  185.589996  181.94

In [322]:
CBA_Stock.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Target_t1,Target_t7
0,2015-01-02,84.959686,85.277962,84.661308,85.277962,51.295269,949439,1,0
1,2015-01-05,85.238182,85.775269,85.049202,85.486832,51.420902,1351651,0,0
2,2015-01-06,84.641411,85.337639,84.412651,84.840332,51.032032,2477275,0,0
3,2015-01-07,84.850281,85.029312,84.094376,84.651360,50.918358,2127190,1,0
4,2015-01-08,85.079041,85.188446,84.671249,84.929848,51.085873,1997761,1,0


In [323]:
CBA_Stock["Return"] = CBA_Stock["Close"].pct_change()
CBA_Stock["LogReturn"] = np.log(CBA_Stock["Close"] / CBA_Stock["Close"].shift(1))

CBA_Stock["Return_lag1"] = CBA_Stock["Return"].shift(1)
CBA_Stock["Return_lag2"] = CBA_Stock["Return"].shift(2)
CBA_Stock["Return_lag3"] = CBA_Stock["Return"].shift(3)
CBA_Stock["Return_lag5"] = CBA_Stock["Return"].shift(5)

CBA_Stock["RollingMean_5"] = CBA_Stock["Return"].rolling(5).mean()
CBA_Stock["RollingStd_5"] = CBA_Stock["Return"].rolling(5).std()

CBA_Stock["RollingMean_10"] = CBA_Stock["Return"].rolling(10).mean()
CBA_Stock["RollingStd_10"] = CBA_Stock["Return"].rolling(10).std()

CBA_Stock["VolumeChange"] = CBA_Stock["Volume"].pct_change()
CBA_Stock["VolumeMA_5"] = CBA_Stock["Volume"].rolling(5).mean()

print(CBA_Stock.shape)
print(CBA_Stock.isna().sum())

(2856, 21)
Date               0
Open               0
High               0
Low                0
Close              0
Adj Close          0
Volume             0
Target_t1          0
Target_t7          0
Return             1
LogReturn          1
Return_lag1        2
Return_lag2        3
Return_lag3        4
Return_lag5        6
RollingMean_5      5
RollingStd_5       5
RollingMean_10    10
RollingStd_10     10
VolumeChange       3
VolumeMA_5         4
dtype: int64


In [324]:
import numpy as np

window = 14

delta = CBA_Stock["Close"].diff()

gain = (delta.where(delta > 0, 0)).rolling(window).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window).mean()

rs = gain / loss
CBA_Stock["RSI_14"] = 100 - (100 / (1 + rs))

In [325]:
ema_12 = CBA_Stock["Close"].ewm(span=12, adjust=False).mean()
ema_26 = CBA_Stock["Close"].ewm(span=26, adjust=False).mean()

CBA_Stock["MACD"] = ema_12 - ema_26
CBA_Stock["MACD_signal"] = CBA_Stock["MACD"].ewm(span=9, adjust=False).mean()

In [326]:
CBA_Stock["BB_Mid"] = CBA_Stock["Close"].rolling(20).mean()
CBA_Stock["BB_Std"] = CBA_Stock["Close"].rolling(20).std()

CBA_Stock["BB_Upper"] = CBA_Stock["BB_Mid"] + 2 * CBA_Stock["BB_Std"]
CBA_Stock["BB_Lower"] = CBA_Stock["BB_Mid"] - 2 * CBA_Stock["BB_Std"]

In [327]:
# Tenkan-sen
CBA_Stock["Tenkan_sen"] = (
    CBA_Stock["High"].rolling(9).max() +
    CBA_Stock["Low"].rolling(9).min()
) / 2

# Kijun-sen
CBA_Stock["Kijun_sen"] = (
    CBA_Stock["High"].rolling(26).max() +
    CBA_Stock["Low"].rolling(26).min()
) / 2

# NO forward shift (critical)
CBA_Stock["Senkou_A"] = (CBA_Stock["Tenkan_sen"] + CBA_Stock["Kijun_sen"]) / 2

CBA_Stock["Senkou_B"] = (
    CBA_Stock["High"].rolling(52).max() +
    CBA_Stock["Low"].rolling(52).min()
) / 2

In [329]:
CBA_Stock = CBA_Stock.dropna().reset_index(drop=True)

print("Final shape:", CBA_Stock.shape)
print("NaNs left:\n", CBA_Stock.isna().sum())

Final shape: (2803, 32)
NaNs left:
 Date              0
Open              0
High              0
Low               0
Close             0
Adj Close         0
Volume            0
Target_t1         0
Target_t7         0
Return            0
LogReturn         0
Return_lag1       0
Return_lag2       0
Return_lag3       0
Return_lag5       0
RollingMean_5     0
RollingStd_5      0
RollingMean_10    0
RollingStd_10     0
VolumeChange      0
VolumeMA_5        0
RSI_14            0
MACD              0
MACD_signal       0
BB_Mid            0
BB_Std            0
BB_Upper          0
BB_Lower          0
Tenkan_sen        0
Kijun_sen         0
Senkou_A          0
Senkou_B          0
dtype: int64


In [330]:
train_df = CBA_Stock[CBA_Stock["Date"] < "2022-01-01"]
val_df   = CBA_Stock[(CBA_Stock["Date"] >= "2022-01-01") & (CBA_Stock["Date"] < "2024-01-01")]
test_df  = CBA_Stock[CBA_Stock["Date"] >= "2024-01-01"]

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (1722, 32)
Val: (503, 32)
Test: (578, 32)


In [333]:
TARGET = "Target_t1"   # we start with t+1 first

FEATURES = [col for col in CBA_Stock.columns if col not in ["Date", "Target_t1", "Target_t7"]]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print("Feature count:", len(FEATURES))

Feature count: 29


In [335]:
inf_counts = np.isinf(X_train).sum()

print("Columns with infinity:")
print(inf_counts[inf_counts > 0])


Columns with infinity:
VolumeChange    4
dtype: int64


In [336]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val   = X_val.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

# Check NaNs after replacing inf
print("Train NaNs:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

print("Val NaNs:")
print(X_val.isna().sum()[X_val.isna().sum() > 0])

print("Test NaNs:")
print(X_test.isna().sum()[X_test.isna().sum() > 0])

Train NaNs:
VolumeChange    4
dtype: int64
Val NaNs:
Series([], dtype: int64)
Test NaNs:
Series([], dtype: int64)


In [337]:
train_valid = X_train.notna().all(axis=1)
val_valid   = X_val.notna().all(axis=1)
test_valid  = X_test.notna().all(axis=1)

X_train = X_train[train_valid]
y_train = y_train[train_valid]

X_val = X_val[val_valid]
y_val = y_val[val_valid]

X_test = X_test[test_valid]
y_test = y_test[test_valid]

print("Cleaned shapes:")
print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Cleaned shapes:
Train: (1718, 29) (1718,)
Val: (503, 29) (503,)
Test: (578, 29) (578,)


In [338]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)